# College Station, TX LVT Shift Notebook

This notebook models a Land Value Tax shift for College Station, Texas, using data from
the Brazos Central Appraisal District (Brazos CAD).
Reference spec: `examples/skills/college_station/FETCH.md`.

**Data sources:**
- Geometry: 2025 Certified Parcel Shapefile (Brazos CAD)
- Valuation: 2025 Certified PACS Export — `APPRAISAL_INFO.TXT` (fixed-width format)
- Zoning: City of College Station open-data FeatureServer

**Policy defaults:**
- Scope: parcels where `SITUS_CITY = 'COLLEGE STATION'`; real property only
- TAMU/state-owned parcels (`ex_exempt = 'T'`) shown as "Exempt / Government" and excluded from LVT modeling
- Reform: split-rate LVT scenarios (2:1, 4:1, 8:1)
- Current tax: sum of CS, BC, and CSISD levies with entity-specific homestead exemptions applied


In [1]:
import io
import sys
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.geometry import Point, Polygon

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "lvt_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from lvt_utils import (
    calculate_category_tax_summary,
    model_split_rate_tax,
    print_category_tax_summary,
)
from policy_analysis import (
    analyze_land_by_improvement_share,
    analyze_parking_lots,
    analyze_vacant_land,
    print_parking_analysis_summary,
    print_vacant_land_summary,
)
from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from viz import create_quintile_summary

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)


In [2]:
data_dir = REPO_ROOT / "examples" / "data" / "college_station"
data_dir.mkdir(parents=True, exist_ok=True)

# ── Data source URLs ──────────────────────────────────────────────────────────
SHP_URL_2025   = "https://brazoscad.org/wp-content/uploads/2025/11/Public_Parcel_Boundary_certified.zip"
PACS_URL_2025  = "https://brazoscad.org/wp-content/uploads/2025/08/2025-CERTIFIED-EXPORT.zip"
ZONING_URL     = "https://maps.cstx.gov/cstx/rest/services/OpenData_PDS/FeatureServer/14/query"

CITY_FILTER_VALUE  = "COLLEGE STATION"
BRAZOS_COUNTY_FIPS = "48041"

# ── 2025 Adopted tax rates (per $100 of assessed value) ──────────────────────
# Source: https://brazoscad.org/tax-information/adopted-tax-rates/
MILLAGE_CS    = 0.511872   # City of College Station
MILLAGE_BC    = 0.419700   # Brazos County
MILLAGE_CSISD = 0.975300   # College Station ISD
# ESDs (0.02–0.09 per $100) and MUDs (~$1.00) apply only to parcels within
# those geographic sub-districts. Incorporate via APPRAISAL_ENTITY_INFO.TXT
# if full-stack accuracy is needed.

# ── Entity-specific exemption amounts (2025) ─────────────────────────────────
# City of College Station
CS_HOMESTEAD_RATE = 0.20     # 20% of appraised value
CS_OV65_AMT       = 30_000  # flat dollar, over-65

# Brazos County — verify optional exemptions at brazoscountytx.gov
BC_HOMESTEAD_AMT  = 0        # TODO: update if BC offers optional homestead
BC_OV65_AMT       = 0        # TODO: update if BC offers over-65 flat

# College Station ISD — mandatory per Tex. Ed. Code §11.13(b)
CSISD_HOMESTEAD_AMT = 40_000
CSISD_OV65_AMT      = 10_000  # additional local over-65 exemption

# ── PACS APPRAISAL_INFO.TXT fixed-width byte offsets (layout v8.0.25) ────────
# Reference: https://brazoscad.org/wp-content/uploads/2021/07/Appraisal-Export-Layout-8.0.25-2.pdf
PROP_COLS = {
    "prop_id":            (1,    12),
    "prop_type_cd":       (13,   17),
    "geo_id":             (547,  596),
    "py_owner_name":      (609,  678),
    "situs_num":          (4460, 4474),
    "situs_street":       (1050, 1099),
    "situs_city":         (1110, 1139),
    "situs_zip":          (1140, 1149),
    "appraised_val":      (1916, 1930),
    "ten_percent_cap":    (1931, 1945),
    "assessed_val":       (1946, 1960),
    "land_hstd_val":      (1796, 1810),
    "land_non_hstd_val":  (1811, 1825),
    "imprv_hstd_val":     (1826, 1840),
    "imprv_non_hstd_val": (1841, 1855),
    "land_acres":         (2772, 2791),
    "hs_exempt":          (2609, 2609),
    "ov65_exempt":        (2610, 2610),
    "ex_exempt":          (2671, 2671),
    "land_state_cd":      (2742, 2751),
    "imprv_state_cd":     (2732, 2741),
}

# ── Cache file paths ──────────────────────────────────────────────────────────
geometry_cache  = data_dir / "cs_geometry_2025cert.parquet"
appraisal_cache = data_dir / "cs_appraisal_info_2025cert.parquet"
zoning_cache    = data_dir / "cs_zoning_2025.parquet"
osm_cache       = data_dir / "cs_osm_surface_parking_2025.parquet"
mapping_cache   = data_dir / "cs_mapping_ready_2025cert.parquet"


In [3]:
def parse_fixed_width(file_obj, col_specs, encoding="latin-1"):
    # Parse a PACS fixed-width text file. file_obj may be a path or open binary stream.
    if hasattr(file_obj, "read"):
        lines = io.TextIOWrapper(file_obj, encoding=encoding).readlines()
    else:
        with open(file_obj, "r", encoding=encoding) as fh:
            lines = fh.readlines()
    rows = [
        {k: ln[s:e].strip() for k, (s, e) in col_specs.items()}
        for ln in lines
        if ln.strip()
    ]
    return pd.DataFrame(rows)


def _pick_parcel_shp(shp_files):
    """Pick the parcel boundary .shp from the zip, avoiding abstract/subdivision layers."""
    def score(name):
        n = name.upper()
        if "ABSTRACT" in n or "SUBDIV" in n or "ABS" in n:
            return 0
        if "PARCEL" in n:
            return 2
        return 1
    return sorted(shp_files, key=score, reverse=True)[0]


def load_shapefile():
    if geometry_cache.exists():
        print(f"Loading geometry cache: {geometry_cache.name}")
        return gpd.read_parquet(geometry_cache)

    print("Downloading 2025 certified shapefile from Brazos CAD...")
    resp = requests.get(SHP_URL_2025, stream=True, timeout=600)
    resp.raise_for_status()
    buf = io.BytesIO()
    for chunk in resp.iter_content(256 * 1024):
        buf.write(chunk)
    buf.seek(0)
    print(f"  Downloaded {buf.tell() / 1e6:.1f} MB")

    import tempfile
    with zipfile.ZipFile(buf) as zf:
        shp_files = [n for n in zf.namelist() if n.lower().endswith(".shp")]
        if not shp_files:
            raise FileNotFoundError(f"No .shp in ZIP. Contents: {zf.namelist()}")
        print(f"  Shapefiles in ZIP: {shp_files}")
        chosen_shp = _pick_parcel_shp(shp_files)
        print(f"  Using: {chosen_shp}")
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            gdf_all = gpd.read_file(Path(tmpdir) / chosen_shp)

    print(f"  Shapefile columns: {gdf_all.columns.tolist()}")

    # Detect SITUS_CITY column (shapefile DBF truncates long names to 10 chars)
    situs_col = next(
        (c for c in gdf_all.columns
         if c.upper().replace("_", "") in ("SITUSCITY", "SITUS_CITY", "SITCITY")),
        None,
    )
    if situs_col is None:
        situs_col = next((c for c in gdf_all.columns if "CITY" in c.upper()), None)

    if situs_col is None:
        print("Warning: no SITUS_CITY column found; keeping all parcels.")
        gdf_cs = gdf_all.copy()
    else:
        gdf_cs = gdf_all[gdf_all[situs_col].str.upper().str.strip() == CITY_FILTER_VALUE].copy()
        print(f"  Filtered on '{situs_col}': {len(gdf_cs):,} College Station parcels")

    # Detect GEO_ID column
    geo_col = next(
        (c for c in gdf_cs.columns
         if c.upper().replace("_", "") in ("GEOID", "GEO_ID", "GEONUM", "GEONO")),
        None,
    )
    if geo_col is None:
        print(f"Warning: GEO_ID column not found. Columns: {gdf_cs.columns.tolist()}")
        print("Inspect the columns above and set geo_col manually if needed.")
    else:
        if geo_col != "GEO_ID":
            gdf_cs = gdf_cs.rename(columns={geo_col: "GEO_ID"})
        print(f"  Join key: '{geo_col}' → 'GEO_ID'")

    if not gdf_cs.crs:
        gdf_cs = gdf_cs.set_crs("EPSG:4326")
    elif gdf_cs.crs.to_epsg() != 4326:
        gdf_cs = gdf_cs.to_crs("EPSG:4326")

    gdf_cs.to_parquet(geometry_cache, index=False)
    print(f"Saved: {geometry_cache.name}  ({len(gdf_cs):,} parcels)")
    return gdf_cs


def load_pacs_appraisal_info():
    if appraisal_cache.exists():
        print(f"Loading PACS appraisal info cache: {appraisal_cache.name}")
        return pd.read_parquet(appraisal_cache)

    pacs_zip_path = data_dir / "_pacs_2025cert_raw.zip"
    if not pacs_zip_path.exists():
        print("Downloading 2025 PACS export (large file, may take several minutes)...")
        with requests.get(PACS_URL_2025, stream=True, timeout=3600) as resp:
            resp.raise_for_status()
            total = int(resp.headers.get("content-length", 0))
            downloaded = 0
            with open(pacs_zip_path, "wb") as f:
                for chunk in resp.iter_content(1024 * 1024):
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total and downloaded % (50 * 1024 * 1024) < 1024 * 1024:
                        print(f"  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB")
        print(f"  Download complete: {downloaded / 1e6:.0f} MB")

    print("Parsing APPRAISAL_INFO.TXT (this may take a minute on the full county file)...")
    with zipfile.ZipFile(pacs_zip_path) as zf:
        names = zf.namelist()
        info_file = next(
            (
                n for n in names
                if "APPRAISAL_INFO" in n.upper()
                and "ENTITY" not in n.upper()
                and "IMPROVEMENT" not in n.upper()
                and "LAND" not in n.upper()
                and "STATE" not in n.upper()
                and "ABSTRACT" not in n.upper()
            ),
            None,
        )
        if info_file is None:
            raise FileNotFoundError(
                "APPRAISAL_INFO.TXT not found in ZIP.\nContents:\n" + "\n".join(names)
            )
        with zf.open(info_file) as f:
            df_all = parse_fixed_width(f, PROP_COLS)

    df_all["_situs_upper"] = df_all["situs_city"].str.upper().str.strip()
    df_cs = df_all[
        (df_all["_situs_upper"] == CITY_FILTER_VALUE)
        & (df_all["prop_type_cd"].str.strip().isin(["R", "M"]))
    ].drop(columns=["_situs_upper"]).copy()

    df_cs.to_parquet(appraisal_cache, index=False)
    print(f"Saved: {appraisal_cache.name}  ({len(df_cs):,} CS real-property records)")
    return df_cs


def fetch_arcgis_geojson(query_url, where="1=1", chunk_size=2000, headers=None):
    session = requests.Session()
    count_resp = session.get(
        query_url,
        params={"f": "json", "where": where, "returnCountOnly": "true"},
        timeout=60,
        headers=headers,
    )
    count_resp.raise_for_status()
    total = int(count_resp.json().get("count", 0))
    rows = []
    for offset in range(0, total, chunk_size):
        params = {
            "f": "geojson",
            "where": where,
            "outFields": "*",
            "resultOffset": offset,
            "resultRecordCount": chunk_size,
            "outSR": 4326,
        }
        resp = session.get(query_url, params=params, timeout=180, headers=headers)
        resp.raise_for_status()
        feats = resp.json().get("features", [])
        if not feats:
            break
        rows.extend(feats)
        print(f"  {min(offset + len(feats), total):,} / {total:,}")
    if not rows:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")
    return gpd.GeoDataFrame.from_features(rows, crs="EPSG:4326")


def load_zoning_layer():
    if zoning_cache.exists():
        print(f"Loading zoning cache: {zoning_cache.name}")
        return gpd.read_parquet(zoning_cache)

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; LVTShift/1.0)",
        "Referer": "https://maps.cstx.gov/",
    }
    try:
        zoning_gdf = fetch_arcgis_geojson(ZONING_URL, where="1=1", chunk_size=2000, headers=headers)
    except Exception as exc:
        print(f"Primary zoning endpoint failed ({exc}); trying open-data GeoJSON fallback.")
        fallback = "https://data-cstx.opendata.arcgis.com/datasets/zoning/geojson"
        resp = requests.get(fallback, timeout=120)
        resp.raise_for_status()
        zoning_gdf = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")

    if len(zoning_gdf) == 0:
        print("Warning: zoning returned zero records.")
        return zoning_gdf

    zoning_gdf.to_parquet(zoning_cache, index=False)
    print(f"Saved: {zoning_cache.name}  ({len(zoning_gdf):,} zones)")
    return zoning_gdf


def fetch_osm_surface_parking(bounds):
    minx, miny, maxx, maxy = bounds
    overpass_eps = [
        "https://overpass-api.de/api/interpreter",
        "https://lz4.overpass-api.de/api/interpreter",
    ]

    def fetch_tile(tx_minx, tx_miny, tx_maxx, tx_maxy):
        query = (
            f"[out:json][timeout:120];"
            f"(way[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx});"
            f"relation[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx});"
            f"node[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx}););out body geom;"
        )
        for ep in overpass_eps:
            for _ in range(3):
                try:
                    r = requests.get(ep, params={"data": query}, timeout=240)
                    r.raise_for_status()
                    return r.json()
                except Exception:
                    pass
        return {"elements": []}

    midx, midy = (minx + maxx) / 2, (miny + maxy) / 2
    tiles = [
        (minx, miny, midx, midy), (midx, miny, maxx, midy),
        (minx, midy, midx, maxy), (midx, midy, maxx, maxy),
    ]
    rows = []
    for tx_minx, tx_miny, tx_maxx, tx_maxy in tiles:
        for el in fetch_tile(tx_minx, tx_miny, tx_maxx, tx_maxy).get("elements", []):
            tags = el.get("tags", {})
            geom = None
            if "geometry" in el and len(el["geometry"]) >= 3:
                try:
                    geom = Polygon([(p["lon"], p["lat"]) for p in el["geometry"]])
                except Exception:
                    pass
            elif "lat" in el and "lon" in el:
                geom = Point(el["lon"], el["lat"])
            if geom:
                rows.append({
                    "osm_id": el.get("id"), "osm_type": el.get("type"),
                    "name": tags.get("name"), "geometry": geom,
                })
    if not rows:
        return gpd.GeoDataFrame(
            columns=["osm_id", "osm_type", "name", "geometry"],
            geometry="geometry", crs="EPSG:4326",
        )
    return (
        gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
        .drop_duplicates(subset=["osm_type", "osm_id"])
        .reset_index(drop=True)
    )


def load_osm_surface_parking(bounds):
    if osm_cache.exists():
        print(f"Loading OSM parking cache: {osm_cache.name}")
        return gpd.read_parquet(osm_cache)
    osm_gdf = fetch_osm_surface_parking(bounds)
    osm_gdf.to_parquet(osm_cache, index=False)
    print(f"Saved: {osm_cache.name}  ({len(osm_gdf):,} lots)")
    return osm_gdf


## Step 1: Load and merge College Station parcel data

In [4]:
gdf_geom = load_shapefile()
df_prop  = load_pacs_appraisal_info()

# Normalise join key
df_prop["_geo_key"]  = df_prop["geo_id"].str.upper().str.strip()
display(gdf_geom.columns.tolist())

gdf_geom["_geo_key"] = gdf_geom["GEO_ID"].str.upper().str.strip()

gdf = gdf_geom.merge(df_prop, on="_geo_key", how="inner")
gdf = gdf.drop(columns=["_geo_key"])
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
print(f"Merged parcel count: {len(gdf):,}")

# Numeric conversions
for col in [
    "appraised_val", "assessed_val",
    "land_hstd_val", "land_non_hstd_val",
    "imprv_hstd_val", "imprv_non_hstd_val",
    "land_acres",
]:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce").fillna(0)

gdf["total_land_val"]  = gdf["land_hstd_val"]  + gdf["land_non_hstd_val"]
gdf["total_imprv_val"] = gdf["imprv_hstd_val"] + gdf["imprv_non_hstd_val"]

# OSM surface-parking spatial join
osm_parking = load_osm_surface_parking(gdf.total_bounds)
if len(osm_parking) > 0:
    gdf_3857 = gdf.to_crs(epsg=3857)
    osm_3857 = osm_parking.to_crs(epsg=3857)
    joined   = gpd.sjoin(
        gdf_3857[["GEO_ID", "geometry"]],
        osm_3857[["osm_id", "geometry"]],
        how="left", predicate="intersects",
    )
    osm_geo_ids = set(joined.loc[joined["osm_id"].notna(), "GEO_ID"])
    gdf["is_osm_surface_parking"] = gdf["GEO_ID"].isin(osm_geo_ids)
else:
    gdf["is_osm_surface_parking"] = False

print(f"OSM surface-parking tagged: {int(gdf['is_osm_surface_parking'].sum()):,}")
display(gdf[[
    "GEO_ID", "prop_type_cd", "total_land_val", "total_imprv_val",
    "appraised_val", "assessed_val", "ex_exempt",
]].head(5))


  Downloaded 0.0 MB
  Shapefiles in ZIP: ['Public_Parcel_Boundary_certified_2025/Abstract.shp', 'Public_Parcel_Boundary_certified_2025/Abstract_Name.shp', 'Public_Parcel_Boundary_certified_2025/Abstract_Number.shp', 'Public_Parcel_Boundary_certified_2025/Address_Number.shp', 'Public_Parcel_Boundary_certified_2025/Block_Number.shp', 'Public_Parcel_Boundary_certified_2025/Centerline_Text.shp', 'Public_Parcel_Boundary_certified_2025/City_Boundary.shp', 'Public_Parcel_Boundary_certified_2025/County_Boundary.shp', 'Public_Parcel_Boundary_certified_2025/Creek.shp', 'Public_Parcel_Boundary_certified_2025/ISD_Boundary.shp', 'Public_Parcel_Boundary_certified_2025/Lake.shp', 'Public_Parcel_Boundary_certified_2025/Lot_Line.shp', 'Public_Parcel_Boundary_certified_2025/Lot_Line_1.shp', 'Public_Parcel_Boundary_certified_2025/Lot_Number.shp', 'Public_Parcel_Boundary_certified_2025/Map_Grid_BN.shp', 'Public_Parcel_Boundary_certified_2025/Map_Grid_BZ.shp', 'Public_Parcel_Boundary_certified_2025/Map_Gri

['FeatureID',
 'ZOrder',
 'Annotation',
 'SymbolID',
 'Status',
 'TextString',
 'FontName',
 'FontSize',
 'Bold',
 'Italic',
 'Underline',
 'VerticalAl',
 'Horizontal',
 'XOffset',
 'YOffset',
 'Angle',
 'FontLeadin',
 'WordSpacin',
 'CharacterW',
 'CharacterS',
 'FlipAngle',
 'Override',
 'CENTROID_X',
 'CENTROID_Y',
 'CENTROID_Z',
 'CENTROID_M',
 'geometry']

KeyError: 'GEO_ID'

## Step 1b: Download zoning layer and stamp parcels

In [ ]:
zoning_gdf = load_zoning_layer()

if len(zoning_gdf) > 0:
    gdf_3857    = gdf.to_crs(epsg=3857)
    zoning_3857 = zoning_gdf.to_crs(epsg=3857)

    gdf_3857["PARCEL_OID"] = gdf_3857["GEO_ID"]
    gdf_3857["centroid"]   = gdf_3857.geometry.centroid
    parcel_pts = gpd.GeoDataFrame(
        gdf_3857.drop(columns=["geometry"]), geometry="centroid", crs="EPSG:3857"
    )

    if "OBJECTID" in zoning_3857.columns:
        zoning_3857 = zoning_3857.rename(columns={"OBJECTID": "ZONING_OID"})
    else:
        zoning_3857["ZONING_OID"] = np.arange(len(zoning_3857), dtype=int)

    join_cols = [c for c in zoning_3857.columns if c != "geometry"]
    stamped   = gpd.sjoin(
        parcel_pts, zoning_3857[join_cols + ["geometry"]], how="left", predicate="within"
    )

    # Detect zoning-district text field
    zone_candidates = [
        "ZONING", "ZONE", "ZONE_NAME", "ZONETYPE", "ZONE_TYPE",
        "DISTRICT", "DISTRICTNAME", "LABEL", "NAME",
    ]
    chosen = next((c for c in zone_candidates if c in stamped.columns), None)
    if chosen is None:
        obj_cols = [c for c in join_cols if stamped[c].dtype == "object"]
        chosen   = obj_cols[0] if obj_cols else None

    if chosen:
        zone_map = stamped.groupby("PARCEL_OID")[chosen].first()
        gdf["ZONING_CLASS"] = gdf["GEO_ID"].map(zone_map)
        print(f"Zoning field used: '{chosen}'")
    else:
        gdf["ZONING_CLASS"] = None
        print("Warning: no zoning text field detected; ZONING_CLASS left null.")
else:
    gdf["ZONING_CLASS"] = None

print(f"Parcels stamped with zoning: {int(gdf['ZONING_CLASS'].notna().sum()):,}")
display(gdf[["GEO_ID", "imprv_state_cd", "ZONING_CLASS"]].head(10))


## Step 2: Quality checks and entity-specific current tax

In [ ]:
# Valuation consistency
gdf["val_check"] = gdf["total_land_val"] + gdf["total_imprv_val"]
gdf["val_diff"]  = (gdf["appraised_val"] - gdf["val_check"]).abs()

taxable = gdf[gdf["appraised_val"] > 0]
print(f"Parcels with appraised_val > 0: {len(taxable):,}")
print(f"Max land+imprv vs appraised mismatch: ${taxable['val_diff'].max():,.0f}")
print(f"Median mismatch:                      ${taxable['val_diff'].median():,.0f}")
# Note: large mismatches occur on ag-deferred or timber parcels — expected.

n_exempt = (gdf["ex_exempt"] == "T").sum()
print(f"\nFully exempt (TAMU/state/church, ex_exempt='T'): {n_exempt:,} of {len(gdf):,}")

# Improvement ratio
gdf["IR"] = np.where(
    gdf["appraised_val"] > 0,
    gdf["total_imprv_val"] / gdf["appraised_val"],
    0,
)

# Entity-specific current tax
# City of College Station: 20% homestead + $30k over-65
cs_hs = np.where(gdf["hs_exempt"] == "T", CS_HOMESTEAD_RATE * gdf["appraised_val"], 0)
cs_o65 = np.where(gdf["ov65_exempt"] == "T", CS_OV65_AMT, 0)
gdf["cs_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - cs_hs - cs_o65),
)

# Brazos County: update BC_HOMESTEAD_AMT / BC_OV65_AMT if county offers optional exemptions
bc_hs  = np.where(gdf["hs_exempt"] == "T", BC_HOMESTEAD_AMT, 0)
bc_o65 = np.where(gdf["ov65_exempt"] == "T", BC_OV65_AMT, 0)
gdf["bc_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - bc_hs - bc_o65),
)

# College Station ISD: $40k mandatory homestead + $10k additional over-65
csisd_hs  = np.where(gdf["hs_exempt"] == "T", CSISD_HOMESTEAD_AMT, 0)
csisd_o65 = np.where(gdf["ov65_exempt"] == "T", CSISD_OV65_AMT, 0)
gdf["csisd_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - csisd_hs - csisd_o65),
)

gdf["current_tax_cs"]    = gdf["cs_taxable"]    * MILLAGE_CS    / 100
gdf["current_tax_bc"]    = gdf["bc_taxable"]    * MILLAGE_BC    / 100
gdf["current_tax_csisd"] = gdf["csisd_taxable"] * MILLAGE_CSISD / 100
gdf["current_tax"]       = gdf["current_tax_cs"] + gdf["current_tax_bc"] + gdf["current_tax_csisd"]

print("\nCombined levy modeled (CS + BC + CSISD):")
print(f"  Total:  ${gdf['current_tax'].sum():>15,.0f}")
print(f"  CS:     ${gdf['current_tax_cs'].sum():>15,.0f}")
print(f"  BC:     ${gdf['current_tax_bc'].sum():>15,.0f}")
print(f"  CSISD:  ${gdf['current_tax_csisd'].sum():>15,.0f}")

display(gdf[["assessed_val", "cs_taxable", "bc_taxable", "csisd_taxable", "current_tax"]].describe())


In [ ]:
gdf["ZONING_CLASS"].value_counts()

## Step 3: Property categories for summary output

In [ ]:
def map_property_category(row) -> str:
    ex  = str(row.get("ex_exempt",      "")).strip().upper()
    isc = str(row.get("imprv_state_cd", "")).strip().upper()
    lsc = str(row.get("land_state_cd",  "")).strip().upper()

    if ex == "T":
        return "Exempt / Government"

    for code in [isc, lsc]:
        if not code:
            continue
        if code.startswith("A"):              # A1=SF, A2=mobile home
            return "Residential"
        if code in ("B1", "B2", "B3", "B4"):  # multi-family
            return "Residential"
        if code in ("C1", "C2"):              # vacant lot / commercial vacant
            return "Vacant / Undeveloped"
        if code.startswith("D"):              # ag / open space
            return "Agricultural"
        if code == "F2":
            return "Industrial"
        if code.startswith("F"):              # F1 commercial real
            return "Commercial"
        if code.startswith(("G", "J", "X")):  # gov / utility / exempt
            return "Exempt / Government"
        if code.startswith("O"):              # residential inventory
            return "Residential"
    return "Other"


def map_refined_category(base: str, imprv_state_cd: str, zoning_class: str) -> str:
    isc = str(imprv_state_cd).strip().upper()
    z   = str(zoning_class).strip().upper()

    if base in ("Exempt / Government", "Vacant / Undeveloped", "Agricultural",
                "Commercial", "Industrial"):
        return base

    if base == "Residential":
        if isc == "B1":
            # Student housing: large apartments in MF / T / PDD zones near TAMU
            if any(tok in z for tok in ["MF", "PDD", "P-MUD"]) or z in ("T", "MF"):
                return "Residential - Student Housing"
            return "Residential - High Density"
        if isc in ("B2", "B3", "B4"):
            return "Residential - Medium Density"
        if any(tok in z for tok in ["GS", "RS", "R-1", "R1"]):
            return "Residential - Low Density"
        if z.startswith("R-") or z == "R":
            return "Residential - Low Density"
        return "Residential - Other"

    if any(tok in z for tok in ["MH", "MU", "WPC"]):
        return "Mixed Use"

    return base


gdf["PROPERTY_CATEGORY"] = gdf.apply(map_property_category, axis=1)
gdf["ANALYSIS_CATEGORY"] = gdf.apply(
    lambda r: map_refined_category(
        r["PROPERTY_CATEGORY"], r.get("imprv_state_cd", ""), r.get("ZONING_CLASS", "")
    ),
    axis=1,
)

display(gdf["ANALYSIS_CATEGORY"].value_counts().to_frame("parcel_count"))

analysis_gdf = gdf[
    gdf["PROPERTY_CATEGORY"].ne("Exempt / Government")
    & gdf["PROPERTY_CATEGORY"].ne("Agricultural")
].copy()
print(f"\nExcluded exempt/ag: {len(gdf) - len(analysis_gdf):,}")
print(f"Parcels for LVT modeling: {len(analysis_gdf):,}")


## Step 3b: Export mapping-ready GeoParquet

In [ ]:
export_cols = [
    "GEO_ID", "prop_id", "py_owner_name",
    "imprv_state_cd", "land_state_cd",
    "PROPERTY_CATEGORY", "ANALYSIS_CATEGORY", "ZONING_CLASS",
    "is_osm_surface_parking",
    "land_acres", "total_land_val", "total_imprv_val",
    "appraised_val", "assessed_val",
    "current_tax",
    "geometry",
]
export_gdf = analysis_gdf[[c for c in export_cols if c in analysis_gdf.columns]].copy()
export_gdf = export_gdf.to_crs(epsg=4326)
export_gdf.to_parquet(mapping_cache, index=False)
print(f"Wrote mapping GeoParquet: {mapping_cache}")
print(f"CRS: {export_gdf.crs}  |  Rows: {len(export_gdf):,}")
display(export_gdf.head(5))


## Step 4: Revenue-neutral split-rate scenarios

In [ ]:
model_df        = analysis_gdf.copy()
current_revenue = model_df["current_tax"].sum()
print(f"Current combined levy (CS + BC + CSISD): ${current_revenue:,.0f}")

scenario_outputs = {}
for ratio in [2, 4, 8]:
    land_mill, imp_mill, revenue, scenario_df = model_split_rate_tax(
        df=model_df.copy(),
        land_value_col="total_land_val",
        improvement_value_col="total_imprv_val",
        current_revenue=current_revenue,
        land_improvement_ratio=ratio,
    )
    new_col = f"tax_shift_{ratio}to1"
    scenario_df[new_col] = scenario_df["new_tax"] - scenario_df["current_tax"]
    scenario_outputs[ratio] = {
        "land_mill": land_mill, "imp_mill": imp_mill,
        "revenue": revenue, "df": scenario_df, "new_col": new_col,
    }
    print(f"{ratio}:1  land mill={land_mill:.4f}  imp mill={imp_mill:.4f}  revenue=${revenue:,.0f}")


## Step 5: Tax-capacity split model and underdevelopment analysis

In [ ]:
cs_city = analysis_gdf.copy()
cs_city["TaxCapacity"]              = cs_city["assessed_val"]
cs_city["TaxCapacity_Improvements"] = cs_city["IR"] * cs_city["TaxCapacity"]
cs_city["TaxCapacity_Land"]         = (1 - cs_city["IR"]) * cs_city["TaxCapacity"]
current_revenue_tc                  = cs_city["current_tax"].sum()

print("Tax Capacity Split Summary:")
total_tc = cs_city["TaxCapacity"].sum()
print(f"  Total Tax Capacity:          ${total_tc:>15,.0f}")
print(f"  Tax Capacity (Improvements): ${cs_city['TaxCapacity_Improvements'].sum():>15,.0f}")
print(f"  Tax Capacity (Land):         ${cs_city['TaxCapacity_Land'].sum():>15,.0f}")
print(f"  Land % of Tax Capacity:      {cs_city['TaxCapacity_Land'].sum() / total_tc * 100:.1f}%")

tc_ratio = 2
tc_land_mill, tc_imp_mill, tc_revenue, cs_city = model_split_rate_tax(
    df=cs_city,
    land_value_col="TaxCapacity_Land",
    improvement_value_col="TaxCapacity_Improvements",
    current_revenue=current_revenue_tc,
    land_improvement_ratio=tc_ratio,
)
cs_city["new_tax_tc"]        = cs_city["new_tax"]
cs_city["tax_change_tc"]     = cs_city["new_tax_tc"] - cs_city["current_tax"]
cs_city["tax_change_pct_tc"] = np.where(
    cs_city["current_tax"] > 0,
    cs_city["tax_change_tc"] / cs_city["current_tax"] * 100,
    0,
)

print(f"\nTC Split-Rate ({tc_ratio}:1)")
print(f"  Land millage:    {tc_land_mill:.6f}")
print(f"  Imp millage:     {tc_imp_mill:.6f}")
print(f"  Revenue neutral: {abs(current_revenue_tc - cs_city['new_tax_tc'].sum()) < 1}")

# Underdeveloped land analysis
cs_city["LAND_DEV_CATEGORY"] = np.where(cs_city["IR"] == 0, "Vacant Land", "Developed")
vacant_results = analyze_vacant_land(
    df=cs_city,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_DEV_CATEGORY",
    vacant_identifier="Vacant Land",
)
underdeveloped = analyze_land_by_improvement_share(
    df=cs_city,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
)
total_land_value = cs_city["total_land_val"].sum()

print(f"\nUndeveloped and Underdeveloped Land")
print(f"  Total non-exempt land value: ${total_land_value:,.0f}")
if "error" not in vacant_results:
    print(
        f"  Vacant (IR=0): {vacant_results['total_vacant_parcels']:,} parcels, "
        f"${vacant_results['total_vacant_land_value']:,.0f} "
        f"({vacant_results['vacant_land_pct_of_total']:.1f}% of non-exempt land value)"
    )
print("\n  By improvement share:")
for row in underdeveloped["categories"]:
    print(
        f"    {row['category']:<35} {row['parcel_count']:>6,} parcels  "
        f"${row['adjusted_land_value']:>14,.0f}  ({row['share_of_total_land_value_pct']:.1f}%)"
    )


## Step 6: Category summary & charts

In [ ]:
primary = scenario_outputs[4]["df"].copy()
primary["tax_change"]     = primary["new_tax"] - primary["current_tax"]
primary["tax_change_pct"] = np.where(
    primary["current_tax"] > 0,
    primary["tax_change"] / primary["current_tax"] * 100,
    0,
)

output_summary = calculate_category_tax_summary(
    df=primary,
    category_col="ANALYSIS_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
)
print_category_tax_summary(output_summary)

plot_df = (
    output_summary[output_summary["property_count"] > 50]
    .sort_values("median_tax_change_pct")
    .copy()
)
fig_height = max(5, len(plot_df) * 0.7)
plt.figure(figsize=(12, fig_height))
bar_colors = np.where(plot_df["median_tax_change_pct"] < 0, "#228B22", "#8B0000")
plt.barh(plot_df["ANALYSIS_CATEGORY"], plot_df["median_tax_change_pct"], color=bar_colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Median tax change % by property category (4:1 split-rate)")
plt.xlabel("Median tax change (%)")
plt.tight_layout()
plt.show()


In [ ]:
summary_filtered = output_summary[output_summary["property_count"] > 50].copy()
summary_sorted   = summary_filtered.sort_values("pct_increase_gt_threshold", ascending=True)

cats      = summary_sorted["ANALYSIS_CATEGORY"].tolist()
pct_inc   = summary_sorted["pct_increase_gt_threshold"].tolist()
pct_dec   = summary_sorted["pct_decrease_gt_threshold"].tolist()
pct_inc_i = [int(round(x)) for x in pct_inc]
pct_dec_i = [int(round(x)) for x in pct_dec]

y = np.arange(len(cats))
fig, ax = plt.subplots(figsize=(8, max(5, len(cats) * 0.6)))

color_inc = "#8B0000"
color_dec = "#228B22"

ax.barh(y, [-v for v in pct_dec], color=color_dec, edgecolor="none", height=0.7)
ax.barh(y, pct_inc, color=color_inc, edgecolor="none", height=0.7)

for i, (inc, dec) in enumerate(zip(pct_inc_i, pct_dec_i)):
    if dec > 0:
        ax.text(-dec - 2, y[i], f"{dec}%", va="center", ha="right", fontsize=8, color=color_dec)
    if inc > 0:
        ax.text(inc + 2, y[i], f"{inc}%", va="center", ha="left",  fontsize=8, color=color_inc)

max_val = max(max(pct_inc), max(pct_dec)) if pct_inc else 10
for i, (cat, inc) in enumerate(zip(cats, pct_inc)):
    ax.text(
        inc + 18 if inc > 0 else 18, y[i], cat,
        va="center", ha="left", fontsize=9, fontweight="bold", color="#222",
    )

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(-max_val - 20, max_val + 50)

title_y = len(cats) - 0.2
ax.text(-max_val * 0.45, title_y, "Percent of parcels\ndecreasing >10%",
        ha="center", va="bottom", fontsize=10)
ax.text( max_val * 0.45, title_y, "Percent of parcels\nincreasing >10%",
        ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


## Step 7: Land-use diagnostics (vacancy, parking, low-improvement share)

In [ ]:
analysis_df = primary.copy()
analysis_df["LAND_USE_FOR_ANALYSIS"] = np.select(
    [
        analysis_df["PROPERTY_CATEGORY"].str.contains("Vacant", na=False),
        analysis_df["is_osm_surface_parking"] == True,
    ],
    ["Vacant Land", "Trans - Parking"],
    default="Other",
)

vacant_results = analyze_vacant_land(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_USE_FOR_ANALYSIS",
    vacant_identifier="Vacant Land",
    owner_col="py_owner_name",
)
print_vacant_land_summary(vacant_results)

parking_results = analyze_parking_lots(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_USE_FOR_ANALYSIS",
    parking_identifier="Trans - Parking",
    min_land_value_threshold=50_000,
    max_improvement_ratio=0.10,
)
print_parking_analysis_summary(parking_results)

low_impr_share = analyze_land_by_improvement_share(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
)
display(pd.DataFrame(low_impr_share["categories"]))


## Step 7b: High-impact parcels

In [ ]:
show_cols = [
    "GEO_ID", "py_owner_name", "imprv_state_cd", "ANALYSIS_CATEGORY",
    "total_land_val", "total_imprv_val",
    "current_tax", "new_tax", "tax_change", "tax_change_pct",
]
available = [c for c in show_cols if c in primary.columns]
top_increase = primary.nlargest(20,  "tax_change")[available]
top_decrease = primary.nsmallest(20, "tax_change")[available]

print("Top 20 increases")
display(top_increase)
print("Top 20 decreases")
display(top_decrease)


## Step 8: Census equity analysis

In [ ]:
equity_df = primary.copy().to_crs(epsg=4326)

try:
    census_data, census_boundaries = get_census_data_with_boundaries(
        fips_code=BRAZOS_COUNTY_FIPS,
        year=2022,
    )
    matched = match_to_census_blockgroups(equity_df, census_boundaries, join_type="left")
    matched = matched[matched["median_income"] > 0].copy()

    bg_summary = (
        matched.groupby("std_geoid")
        .agg(
            median_income       =("median_income",    "first"),
            minority_pct        =("minority_pct",     "first"),
            total_current_tax   =("current_tax",      "sum"),
            total_new_tax       =("new_tax",           "sum"),
            mean_tax_change     =("tax_change",        "mean"),
            median_tax_change   =("tax_change",        "median"),
            median_tax_change_pct=("tax_change_pct",  "median"),
            parcel_count        =("GEO_ID",            "count"),
            has_vacant_land     =("ANALYSIS_CATEGORY", lambda x: (x == "Vacant / Undeveloped").any()),
        )
        .reset_index()
    )
    bg_summary = bg_summary[bg_summary["median_income"] > 0].copy()
    bg_summary["mean_tax_change_pct"] = np.where(
        bg_summary["total_current_tax"] > 0,
        (bg_summary["total_new_tax"] - bg_summary["total_current_tax"])
        / bg_summary["total_current_tax"] * 100,
        0,
    )
    non_vacant_bg = bg_summary[~bg_summary["has_vacant_land"]].copy()

    income_q    = create_quintile_summary(bg_summary,    "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    nv_income_q = create_quintile_summary(non_vacant_bg, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    minority_q  = create_quintile_summary(bg_summary,    "minority_pct",  "minority_pct",  "mean_tax_change", "mean_tax_change_pct")
    nv_minor_q  = create_quintile_summary(non_vacant_bg, "minority_pct",  "minority_pct",  "mean_tax_change", "mean_tax_change_pct")

    display(income_q)
    display(nv_income_q)
    display(minority_q)
    display(nv_minor_q)

    # Line charts
    for q_all, q_nv, x_col, title in [
        (income_q,   nv_income_q, "median_income_quintile",
         "Mean tax change % by neighborhood income quintile"),
        (minority_q, nv_minor_q,  "minority_pct_quintile",
         "Mean tax change % by minority-share quintile"),
    ]:
        plt.figure(figsize=(10, 5))
        plt.plot(q_all[x_col], q_all["mean_tax_change_pct"], marker="o", label="All properties")
        plt.plot(q_nv[x_col],  q_nv["mean_tax_change_pct"],  marker="o",
                 label="Excl. vacant-land neighborhoods")
        plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
        plt.title(title)
        plt.xlabel(x_col.replace("_", " ").title())
        plt.ylabel("Mean tax change (%)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Inverted bar charts (median, excl. vacant)
    sns.set_theme(style="whitegrid", font_scale=1.15)

    def inverted_bar(quintile_df, lbl_col, palette, title):
        fig, ax = plt.subplots(figsize=(10, 6))
        vals      = quintile_df["median_tax_change_pct"]
        labels    = quintile_df[lbl_col]
        colors    = sns.color_palette(palette, n_colors=len(vals))
        color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
        bars      = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
        ax.yaxis.set_visible(False)
        ax.set_title(title, weight="bold", pad=30)
        sns.despine(left=True, right=True, top=True, bottom=True)
        for bar, val in zip(bars, vals):
            ax.annotate(
                f"{val:.1f}%",
                xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha="center", va="center", fontsize=13, fontweight="bold",
            )
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        plt.xticks(fontweight="bold")
        margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
        ax.set_ylim(-margin, margin)
        ax.axhline(y=0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

    inverted_bar(nv_income_q, "median_income_quintile", "Greens",
                 "Median Tax Change by Neighborhood Income (Excl. Vacant)")
    inverted_bar(nv_minor_q,  "minority_pct_quintile",  "Purples",
                 "Median Tax Change by Neighborhood Minority % (Excl. Vacant)")

    # Subgroup quintile charts
    def render_quintile_bars(df_subset, title_prefix):
        if len(df_subset) < 25:
            print(f"Skipping {title_prefix}: too few parcels ({len(df_subset)}).")
            return
        inc   = create_quintile_summary(df_subset, "median_income", "median_income",
                                        "tax_change", "tax_change_pct")
        minor = create_quintile_summary(df_subset, "minority_pct",  "minority_pct",
                                        "tax_change", "tax_change_pct")
        for summ, lbl_col, palette, ttl in [
            (inc,   "median_income_quintile", "Greens",
             f"Median Tax Change by Income ({title_prefix})"),
            (minor, "minority_pct_quintile",  "Purples",
             f"Median Tax Change by Minority % ({title_prefix})"),
        ]:
            inverted_bar(summ, lbl_col, palette, ttl)

    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Low Density",    na=False)],
        "Residential - Low Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Medium Density", na=False)],
        "Residential - Medium Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("High Density",   na=False)],
        "Residential - High Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Student",        na=False)],
        "Residential - Student Housing",
    )
    render_quintile_bars(
        matched[matched["PROPERTY_CATEGORY"].str.contains("Residential",    na=False)],
        "All Residential",
    )

except Exception as exc:
    print("Census equity analysis skipped:", exc)
    print("Set CENSUS_API_KEY in .env and rerun this cell.")
